# Proyecto: Migración de Data Swamp a Data Lakehouse y Análisis Predictivo
**Contexto:** Una fábrica ensambladora de vehículos ha sensorizado toda su línea de producción. Guardan la temperatura, vibración y consumo eléctrico de sus robots cada milisegundo. Tienen un Data Swamp de 50 TB de CSVs sucios.

## 1. Diseño Arquitectónico (ELT vs ETL)

El proceso actual utiliza un paradigma **ETL tradicional** donde Python extrae datos y limpia todo en memoria RAM. Esto es ineficiente y no escala para 50 TB de datos.

La propuesta es migrar a un paradigma **ELT basado en un Data Lakehouse** (como Databricks, Snowflake o un stack open source con Delta Lake/Iceberg + Trino/Spark).

### Justificación de ELT sobre ETL:
1. **Evita cuellos de botella de RAM:** En lugar de cargar todo a un servidor Python, cargamos en crudo (Load) a un almacenamiento barato (Object Storage).
2. **Transformaciones distribuidas:** Se aprovecha el motor de cómputo del Lakehouse (Spark/SQL) para transformar los datos en paralelo.
3. **Arquitectura Medallón:** 
   - **Bronze (Raw):** Datos CSV originales ingestados tal cual.
   - **Silver (Clean/Conformed):** Limpieza de nulos, filtros de '9999', casteos.
   - **Gold (Curated):** Agregaciones para ML o BI.

```mermaid
graph TD
    subgraph Actual: ETL en Python (Limitado por RAM)
        A1[(Data Swamp 50TB CSV)] --> B1[Extracción a Memoria]
        B1 --> C1[Limpieza en Pandas]
        C1 -. Falla por OOM .-> D1[Almacenamiento Final]
    end

    subgraph Propuesto: ELT con Data Lakehouse (Escalable)
        A2[(Data Swamp 50TB CSV)] -->|1. Extract & Load| B2[(Bronze Layer - Raw)]
        B2 -->|2. Transform Distribuido| C2[(Silver Layer - Limpio)]
        C2 -->|3. Agregaciones| D2[(Gold Layer - ML/BI)]
    end
    
    style A1 fill:#ffcccc,stroke:#ff0000
    style C1 fill:#ffcccc,stroke:#ff0000
    style B2 fill:#cce5ff,stroke:#0066cc
    style C2 fill:#cce5ff,stroke:#0066cc
    style D2 fill:#cce5ff,stroke:#0066cc
```

## 2. Código: Optimización de Tipos de Datos (Dtypes)
Simularemos la carga de un dataset masivo. Demostraremos cuánto pesa en RAM usando los tipos de datos por defecto (`float64`, `int64`) y la reducción extrema al convertir a tipos de 8, 16 o 32 bits (Downcasting).

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Simulación de dataset masivo (10 millones de filas)
n_rows = 10_000_000
np.random.seed(42)

print("Generando dataset masivo...")
df = pd.DataFrame({
    'id_sensor': np.random.randint(1, 100, n_rows),
    'temperatura': np.random.uniform(20.0, 100.0, n_rows),
    'vibracion': np.random.uniform(0.0, 5.0, n_rows),
    'consumo_electrico': np.random.uniform(100.0, 500.0, n_rows)
})

# Memoria original
mem_original = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"\nMemoria Original (Por defecto 64-bit): {mem_original:.2f} MB")
print(df.dtypes)

# Optimización (Downcasting)
df_opt = df.copy()
df_opt['id_sensor'] = df_opt['id_sensor'].astype(np.int8) # Valores de 1 a 100 caben en 8 bits
df_opt['temperatura'] = df_opt['temperatura'].astype(np.float32) 
df_opt['vibracion'] = df_opt['vibracion'].astype(np.float32)
df_opt['consumo_electrico'] = df_opt['consumo_electrico'].astype(np.float32)

# Memoria optimizada
mem_optimizada = df_opt.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"\nMemoria Optimizada (8 y 32-bit): {mem_optimizada:.2f} MB")
print(df_opt.dtypes)

reduccion = (1 - (mem_optimizada / mem_original)) * 100
print(f"\nREDUCCIÓN DE MEMORIA: {reduccion:.2f}%")

## 3. Código: Imputación de Series de Tiempo
Simularemos una caída de sensor insertando `NaN` en una serie temporal. Usaremos interpolación matemática (spline/polinomial) para reconstruir la señal sin usar ceros, preservando la tendencia para el modelo.

In [ ]:
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# Crear una serie de tiempo continua (una onda senoidal con ruido)
time = np.arange(0, 100, 0.5)
signal = np.sin(time / 5) + np.random.normal(0, 0.1, len(time))

df_ts = pd.DataFrame({'timestamp': time, 'vibracion': signal})
df_ts = df_ts.set_index('timestamp')

# Simular caída de sensor (insertar NaNs en un bloque)
df_ts_ruined = df_ts.copy()
df_ts_ruined.iloc[40:60] = np.nan # Caída prolongada del sensor

# Imputación usando interpolación 'spline' de orden 3
# Esto permite una curva suave en lugar de una línea recta o relleno con ceros
df_ts_repaired = df_ts_ruined.copy()
df_ts_repaired['vibracion'] = df_ts_repaired['vibracion'].interpolate(method='spline', order=3)

# Visualización
plt.figure(figsize=(14, 6))
plt.plot(df_ts.index, df_ts['vibracion'], label='Señal Original Real', alpha=0.5, color='green', linewidth=2)
plt.plot(df_ts_ruined.index, df_ts_ruined['vibracion'], 'o', label='Datos Disponibles (con Caída)', color='red')
plt.plot(df_ts_repaired.index, df_ts_repaired['vibracion'], '--', label='Señal Reconstruida (Spline O3)', color='blue')
plt.title('Reconstrucción de Señal de Sensor Caído vía Interpolación Spline', fontsize=14)
plt.xlabel('Tiempo')
plt.ylabel('Vibración')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

## 4. Regresión Lineal Múltiple para predecir RUL
Vamos a entrenar un modelo que predice la Vida Útil Remanente (Remaining Useful Life - RUL) de un motor basado en vibración y temperatura. Guardaremos este modelo para luego usarlo en nuestra imagen Docker.

In [ ]:
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Simulación de datos para entrenamiento
n_samples = 5000
np.random.seed(42)

# Características
temp = np.random.normal(80, 15, n_samples)
vib = np.random.normal(2.5, 0.8, n_samples)

# Variable objetivo (RUL): a mayor temperatura y vibración, menor vida útil
# Supongamos una vida útil base de 5000 horas
rul = 5000 - (temp * 15) - (vib * 200) + np.random.normal(0, 100, n_samples)

df_model = pd.DataFrame({'temperatura': temp, 'vibracion': vib, 'RUL': rul})

X = df_model[['temperatura', 'vibracion']]
y = df_model['RUL']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenamiento
modelo_rul = LinearRegression()
modelo_rul.fit(X_train, y_train)

y_pred = modelo_rul.predict(X_test)
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

# Guardar modelo para Docker
joblib.dump(modelo_rul, 'modelo_rul.pkl')
print("Modelo guardado como 'modelo_rul.pkl'")

## 5. Orquestación en Docker
El proceso de limpieza e inferencia debe ejecutarse de forma aislada y reproducible. A continuación, definimos los archivos necesarios para construir la imagen de Docker.

*Estos archivos fueron generados en la carpeta del proyecto:*
- `pipeline.py`: Script que limpia los datos en streaming/batch y ejecuta el modelo.
- `Dockerfile`: Definición de la imagen optimizada (usando python:3.9-slim).
- `requirements.txt`: Dependencias.
